# Scraping Berita Detik.com


## 1.1 Install Dependensi


In [19]:
!pip install requests beautifulsoup4 pandas lxml -q

## 1.2 Import Library


In [20]:
import csv
import time
import random
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime

## 1.3 Konfigurasi Scraping


In [ ]:
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}


KATEGORI = {
    "ekonomi":    "https://finance.detik.com/indeks",
    "teknologi":  "https://inet.detik.com/indeks",
    "olahraga":   "https://sport.detik.com/indeks",
    "politik":    "https://news.detik.com/indeks",
    "kesehatan":  "https://health.detik.com/indeks",
    "hiburan":    "https://hot.detik.com/indeks",
    "otomotif":   "https://oto.detik.com/indeks",
    "travel":     "https://travel.detik.com/indeks",
    "kuliner":    "https://food.detik.com/indeks",
    "gaya_hidup": "https://wolipop.detik.com/indeks",   
    "umum":       "https://news.detik.com/berita/indeks",
}

MAX_HALAMAN   = 55
TARGET_TOTAL  = 8000
DELAY_MIN, DELAY_MAX = 0.5, 1.5
OUTPUT_FILE   = "dataset_berita.csv"

print(f"Kategori  : {len(KATEGORI)}")
print(f"Max halaman per kategori: {MAX_HALAMAN}")
print(f"Estimasi maksimum: {len(KATEGORI) * MAX_HALAMAN * 15:,} artikel")

## 1.4 Fungsi: Ambil Link Artikel dari Halaman Indeks


In [ ]:
def ambil_link_artikel(url_indeks, halaman=1):
    """
    Ambil daftar URL artikel dari halaman indeks Detik.

    PERBAIKAN: Detik menggunakan query-string ?page=N, bukan /{N}.
    """
    url = f"{url_indeks}?page={halaman}"

    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print(f"  [!] HTTP error {e} — {url}")
        return []
    except Exception as e:
        print(f"  [!] Gagal ambil indeks {url}: {e}")
        return []

    soup = BeautifulSoup(r.text, "html.parser")
    links = []

    # Selector utama: elemen <article>
    for article in soup.find_all("article"):
        tag_a = article.find("a", href=True)
        if tag_a and tag_a["href"].startswith("http"):
            links.append(tag_a["href"])

    # Fallback: div list-content__item (beberapa subdomain)
    if not links:
        for div in soup.find_all("div", class_="list-content__item"):
            tag_a = div.find("a", href=True)
            if tag_a and tag_a["href"].startswith("http"):
                links.append(tag_a["href"])

    # Fallback: any <a> whose href contains the subdomain
    if not links:
        domain = url_indeks.split("/")[2]   # e.g. finance.detik.com
        for tag_a in soup.find_all("a", href=True):
            href = tag_a["href"]
            if domain in href and "/read/" in href:
                links.append(href)

    return list(set(links))

## 1.5 Fungsi: Parsing Halaman Artikel


In [ ]:
def parse_artikel(url, kategori):
    """Ekstrak judul, tanggal, dan isi artikel dari halaman detail."""
    try:
        r = requests.get(url, headers=HEADERS, timeout=15)
        r.raise_for_status()
    except Exception as e:
        return None

    soup = BeautifulSoup(r.text, "html.parser")

    # --- Judul ---
    judul_tag = (
        soup.find("h1", class_="detail__title") or
        soup.find("h1", class_="title") or
        soup.find("h1")
    )
    judul = judul_tag.get_text(strip=True) if judul_tag else ""
    if not judul:
        return None

    # --- Tanggal ---
    tanggal_tag = (
        soup.find("div", class_="detail__date") or
        soup.find("div", class_="date") or
        soup.find("span", class_="date")
    )
    tanggal = tanggal_tag.get_text(strip=True) if tanggal_tag else ""

    body = (
        soup.find("div", class_="detail__body-text") or
        soup.find("div", class_="itp_bodycontent") or
        soup.find("div", class_="detail__body") or
        soup.find("div", attrs={"id": "detikdetailtext"}) or
        soup.find("div", class_="read__content") or
        soup.find("article")
    )

    if not body:
        return None

    paragraf = [p.get_text(strip=True) for p in body.find_all("p") if p.get_text(strip=True)]
    isi = " ".join(paragraf)

    if len(isi) < 100:        # buang artikel terlalu pendek
        return None

    return {
        "kategori": kategori,
        "judul": judul,
        "tanggal": tanggal,
        "isi": isi,
        "url": url,
    }

## 1.6 Eksekusi Scraping



In [ ]:
print(f"[+] Mulai scraping {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

semua_data   = []
url_dikunjungi = set()
stop_flag    = False

with open(OUTPUT_FILE, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["kategori", "judul", "tanggal", "isi", "url"])
    writer.writeheader()

for kategori, url_indeks in KATEGORI.items():
    if stop_flag:
        break

    print(f"\n=== Kategori: {kategori.upper()} ===")
    kat_count = 0

    for hal in range(1, MAX_HALAMAN + 1):
        if stop_flag:
            break

        links = ambil_link_artikel(url_indeks, hal)
        links = [l for l in links if l not in url_dikunjungi]

        if not links:
            print(f"  Halaman {hal}: tidak ada link baru, skip sisa halaman.")
            break

        print(f"  Halaman {hal}: {len(links)} link baru")

        for url in links:
            url_dikunjungi.add(url)
            data = parse_artikel(url, kategori)
            if data:
                semua_data.append(data)
                kat_count += 1

                # Simpan incremental ke CSV
                with open(OUTPUT_FILE, "a", newline="", encoding="utf-8") as f:
                    writer = csv.DictWriter(f, fieldnames=["kategori", "judul", "tanggal", "isi", "url"])
                    writer.writerow(data)

            time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

        total_sekarang = len(semua_data)
        print(f"    -> Sub-total {kategori}: {kat_count} | Total: {total_sekarang:,}")

        if total_sekarang >= TARGET_TOTAL:
            print(f"\n[+] Target {TARGET_TOTAL:,} artikel tercapai — scraping dihentikan.")
            stop_flag = True
            break

    time.sleep(random.uniform(2, 5))

print(f"\n[+] Selesai: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"[+] Total artikel terkumpul: {len(semua_data):,}")

[+] Mulai scraping 2026-05-22 19:11:49

=== Kategori: EKONOMI ===
  Halaman 1: 20 link baru
    -> Sub-total ekonomi: 18 | Total: 18
  Halaman 2: 20 link baru
    -> Sub-total ekonomi: 35 | Total: 35
  Halaman 3: 20 link baru
    -> Sub-total ekonomi: 53 | Total: 53
  Halaman 4: 20 link baru
    -> Sub-total ekonomi: 72 | Total: 72
  Halaman 5: 20 link baru
    -> Sub-total ekonomi: 91 | Total: 91
  Halaman 6: 20 link baru
    -> Sub-total ekonomi: 111 | Total: 111
  Halaman 7: 20 link baru
    -> Sub-total ekonomi: 128 | Total: 128
  Halaman 8: 20 link baru
    -> Sub-total ekonomi: 148 | Total: 148
  Halaman 9: 20 link baru
    -> Sub-total ekonomi: 168 | Total: 168
  Halaman 10: 20 link baru
    -> Sub-total ekonomi: 187 | Total: 187
  Halaman 11: 20 link baru
    -> Sub-total ekonomi: 205 | Total: 205
  Halaman 12: 20 link baru
    -> Sub-total ekonomi: 219 | Total: 219
  Halaman 13: 20 link baru
    -> Sub-total ekonomi: 235 | Total: 235
  Halaman 14: 20 link baru
    -> Sub-total

In [ ]:
KATEGORI_BARU = {
    "gaya_hidup": "https://wolipop.detik.com/indeks",
    "umum":       "https://news.detik.com/berita/indeks",
}

TARGET_TOTAL  = 8000
DELAY_MIN, DELAY_MAX = 0.5, 1.5

df_ada = pd.read_csv(OUTPUT_FILE, encoding="utf-8")
url_dikunjungi = set(df_ada["url"].dropna().tolist())
semua_data = df_ada.to_dict("records")

print(f"[+] Data existing : {len(semua_data):,} artikel")
print(f"[+] Kategori ada  : {df_ada['kategori'].unique().tolist()}")
print(f"[+] Kekurangan    : {max(0, TARGET_TOTAL - len(semua_data)):,} artikel\n")

kekurangan = TARGET_TOTAL - len(semua_data)
if kekurangan <= 0:
    print("[+] Target sudah tercapai, tidak perlu scraping tambahan.")
else:
    stop_flag = False

    for kategori, url_indeks in KATEGORI_BARU.items():
        if stop_flag:
            break
        print(f"\n=== Kategori BARU: {kategori.upper()} ===")
        kat_count = 0

        for hal in range(1, 56):   # max 55 halaman per kategori baru
            if stop_flag:
                break

            links = ambil_link_artikel(url_indeks, hal)
            links = [l for l in links if l not in url_dikunjungi]

            if not links:
                print(f"  Halaman {hal}: tidak ada link baru, lanjut kategori berikutnya.")
                break

            print(f"  Halaman {hal}: {len(links)} link baru")

            for url in links:
                url_dikunjungi.add(url)
                data = parse_artikel(url, kategori)
                if data:
                    semua_data.append(data)
                    kat_count += 1
                    # Append ke CSV yang sudah ada
                    with open(OUTPUT_FILE, "a", newline="", encoding="utf-8") as f:
                        writer = csv.DictWriter(f, fieldnames=["kategori","judul","tanggal","isi","url"])
                        writer.writerow(data)
                time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

            total_sekarang = len(semua_data)
            print(f"    -> Sub-total {kategori}: {kat_count} | Total: {total_sekarang:,}")

            if total_sekarang >= TARGET_TOTAL:
                print(f"\n[+] Target {TARGET_TOTAL:,} artikel tercapai!")
                stop_flag = True
                break

        time.sleep(random.uniform(2, 4))

    if len(semua_data) < TARGET_TOTAL and not stop_flag:
        print(f"\n[~] Masih kurang {TARGET_TOTAL - len(semua_data):,}, lanjut top-up kategori lama...")
        KATEGORI_LAMA = {
            "ekonomi":   "https://finance.detik.com/indeks",
            "teknologi": "https://inet.detik.com/indeks",
            "olahraga":  "https://sport.detik.com/indeks",
            "politik":   "https://news.detik.com/indeks",
            "kesehatan": "https://health.detik.com/indeks",
            "hiburan":   "https://hot.detik.com/indeks",
            "otomotif":  "https://oto.detik.com/indeks",
            "travel":    "https://travel.detik.com/indeks",
            "kuliner":   "https://food.detik.com/indeks",
        }
        for kategori, url_indeks in KATEGORI_LAMA.items():
            if stop_flag:
                break
            print(f"\n=== Top-up: {kategori.upper()} ===")
            for hal in range(31, 80):   
                if stop_flag:
                    break
                links = ambil_link_artikel(url_indeks, hal)
                links = [l for l in links if l not in url_dikunjungi]
                if not links:
                    break
                for url in links:
                    url_dikunjungi.add(url)
                    data = parse_artikel(url, kategori)
                    if data:
                        semua_data.append(data)
                        with open(OUTPUT_FILE, "a", newline="", encoding="utf-8") as f:
                            writer = csv.DictWriter(f, fieldnames=["kategori","judul","tanggal","isi","url"])
                            writer.writerow(data)
                    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))
                if len(semua_data) >= TARGET_TOTAL:
                    print(f"[+] Target tercapai!")
                    stop_flag = True
                    break
            time.sleep(random.uniform(1, 3))

print(f"\n[+] Selesai: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"[+] Total akhir: {len(semua_data):,} artikel")

[+] Data existing : 7,454 artikel
[+] Kategori ada  : ['ekonomi', 'olahraga', 'politik', 'kesehatan', 'hiburan', 'otomotif', 'travel', 'kuliner', 'gaya_hidup', 'umum']
[+] Kekurangan    : 546 artikel


=== Kategori BARU: GAYA_HIDUP ===
  Halaman 1: 1 link baru
    -> Sub-total gaya_hidup: 0 | Total: 7,454
  Halaman 2: 2 link baru
    -> Sub-total gaya_hidup: 0 | Total: 7,454
  Halaman 3: 3 link baru
    -> Sub-total gaya_hidup: 0 | Total: 7,454
  Halaman 4: 1 link baru
    -> Sub-total gaya_hidup: 0 | Total: 7,454
  Halaman 5: 2 link baru
    -> Sub-total gaya_hidup: 0 | Total: 7,454
  Halaman 6: 4 link baru
    -> Sub-total gaya_hidup: 0 | Total: 7,454
  Halaman 7: 3 link baru
    -> Sub-total gaya_hidup: 0 | Total: 7,454
  Halaman 8: 1 link baru
    -> Sub-total gaya_hidup: 0 | Total: 7,454
  Halaman 9: 1 link baru
    -> Sub-total gaya_hidup: 0 | Total: 7,454


KeyboardInterrupt: 

## 1.7 Baca CSV dan Verifikasi


In [31]:
df = pd.read_csv(OUTPUT_FILE, encoding="utf-8")
print(f"[+] Shape: {df.shape}")
print(f"[+] Jumlah kategori: {df['kategori'].nunique()}")
df.head()

[+] Shape: (7454, 5)
[+] Jumlah kategori: 10


,kategori,judul,tanggal,isi,url
0,ekonomi,Purbaya Yakin IHSG Balik ke 8.000: Minggu Depa...,"Jumat, 22 Mei 2026 17:05 WIB",Indeks Harga Saham Gabungan (IHSG) sempat meny...,https://finance.detik.com/bursa-dan-valas/d-85...
1,ekonomi,"Dorong Dolar AS Turun ke Rp 15.000, Purbaya: M...","Jumat, 22 Mei 2026 17:30 WIB",Menteri Keuangan Purbaya Yudhi Sadewa menarget...,https://finance.detik.com/bursa-dan-valas/d-85...
2,ekonomi,Tiba-tiba Menteri PU Sebut Sekolah Rakyat Proy...,"Jumat, 22 Mei 2026 16:28 WIB",Menteri Pekerjaan Umum (PU) Dody Hanggodo meni...,https://finance.detik.com/infrastruktur/d-8500...
3,ekonomi,"Usai Bertemu Eks Gubernur BI, Purbaya Pastikan...","Jumat, 22 Mei 2026 17:41 WIB",Menteri Keuangan (Menkeu) Purbaya Yudhi Sadewa...,https://finance.detik.com/berita-ekonomi-bisni...
4,ekonomi,Video: Waspada Love Scam! Warga RI Banyak Jadi...,10 Views | \r\n ...,Otoritas Jasa Keuangan(OJK) mengingatkan masya...,https://20.detik.com/detikupdate/20260522-2605...


## 1.8 Statistik Dataset


In [29]:
print("Distribusi per kategori:")
print(df["kategori"].value_counts())
print(f"\nTotal artikel         : {len(df):,}")
print(f"Jumlah kategori       : {df['kategori'].nunique()}")
print(f"Rata-rata panjang isi : {df['isi'].str.len().mean():.0f} karakter")
print(f"\nCek missing values:")
print(df.isnull().sum())

Distribusi per kategori:
kategori
olahraga      1555
ekonomi       1481
gaya_hidup    1012
otomotif       591
kesehatan      587
kuliner        584
travel         567
politik        538
hiburan        516
umum            23
Name: count, dtype: int64

Total artikel         : 7,454
Jumlah kategori       : 10
Rata-rata panjang isi : 2245 karakter

Cek missing values:
kategori       0
judul          0
tanggal     1017
isi            0
url            0
dtype: int64
